# 04 · Model Training
**Ninja Van Capstone 2026 — Last-Mile Delivery Failure Prediction**

This notebook trains three models:
| Model | Library | Tuning Method |
|---|---|---|
| LightGBM | `lightgbm` | Optuna (50 trials) |
| Random Forest | `sklearn` | GridSearchCV |
| Logistic Regression | `sklearn` | GridSearchCV |

Each model is tuned via 5-fold stratified cross-validation optimising **F1 on the failure class**.

**Outputs:**
- `/content/outputs/model_lgbm.joblib`
- `/content/outputs/model_rf.joblib`
- `/content/outputs/model_lr.joblib`
- `/content/outputs/cv_results.json`

> **Run order:** Run after `03_feature_engineering.ipynb`.


## 1 · Setup & Load Data

In [ ]:
import sys, json, joblib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO_DIR = Path("/content/nv-lastmile-training")
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROC_DIR = Path("/content/data/processed")
OUT_DIR  = Path("/content/outputs")

with open(REPO_DIR / "config" / "config.yaml") as f:
    cfg = yaml.safe_load(f)

RANDOM_STATE  = cfg["data"]["random_state"]
USE_SMOTE     = cfg["features"]["use_smote"]
OPTUNA_TRIALS = cfg["models"]["lgbm"]["optuna_trials"]
TARGET_COL    = cfg["features"]["target_col"]

print(f"Config: random_state={RANDOM_STATE}, use_smote={USE_SMOTE}, optuna_trials={OPTUNA_TRIALS}")

In [ ]:
# Load train split
train_df = pd.read_parquet(PROC_DIR / "features_train.parquet")

# If SMOTE was generated and enabled, use it
smote_path = PROC_DIR / "features_train_smote.parquet"
if USE_SMOTE and smote_path.exists():
    train_df = pd.read_parquet(smote_path)
    print(f"✅ Using SMOTE-balanced training set: {len(train_df):,} rows")
else:
    print(f"✅ Using original training set: {len(train_df):,} rows")

X_train_raw = train_df.drop(columns=[TARGET_COL])
y_train     = train_df[TARGET_COL].astype(int).values

# Load class weights
with open(OUT_DIR / "class_weights.json") as f:
    class_weights = json.load(f)
print(f"Class weights: {class_weights}")
print(f"Failure rate in train: {y_train.mean():.4f}")

In [ ]:
# Load and apply preprocessor
preprocessor = joblib.load(OUT_DIR / "preprocessor.joblib")
X_train = preprocessor.transform(X_train_raw)

with open(OUT_DIR / "feature_names.json") as f:
    feature_names = json.load(f)

print(f"Preprocessor loaded — transformed shape: {X_train.shape}")
print(f"Feature count: {len(feature_names)}")

## 2 · LightGBM — Optuna Hyperparameter Search

> ⏱️ 50 Optuna trials with 5-fold CV. Estimated time: **15–30 minutes** on CPU, **5–10 min** on GPU.

In [ ]:
from train import run_optuna_lgbm, train_lgbm, cross_validate_model

print("🔍 Starting Optuna search for LightGBM...")
print(f"   Trials: {OPTUNA_TRIALS}  |  CV folds: 5  |  Metric: F1 (failure class)")
print("-" * 60)

lgbm_best_params = run_optuna_lgbm(
    X_train, y_train,
    n_trials=OPTUNA_TRIALS,
    random_state=RANDOM_STATE,
)

In [ ]:
# Cross-validate with best params to get reliable CV estimate
import lightgbm as lgb

print("\n📊 Cross-validation with best LightGBM params:")
lgbm_cv_model = lgb.LGBMClassifier(**lgbm_best_params, random_state=RANDOM_STATE,
                                     objective="binary", metric="auc", verbosity=-1)
lgbm_cv_result = cross_validate_model(lgbm_cv_model, X_train, y_train,
                                       model_name="LightGBM", random_state=RANDOM_STATE)

In [ ]:
# Refit on full training set
print("\n🏋️ Fitting LightGBM on full training set...")
lgbm_model = train_lgbm(X_train, y_train, lgbm_best_params)

# Sanity check: training AUC
from sklearn.metrics import roc_auc_score
train_auc = roc_auc_score(y_train, lgbm_model.predict_proba(X_train)[:, 1])
print(f"   Training AUC-ROC (sanity): {train_auc:.4f}")

# Save
lgbm_path = OUT_DIR / "model_lgbm.joblib"
joblib.dump(lgbm_model, lgbm_path, compress=3)
print(f"✅ LightGBM saved → {lgbm_path}")

## 3 · Random Forest — GridSearchCV

In [ ]:
from train import grid_search_rf

rf_param_grid = cfg["models"]["rf"]["param_grid"]
# Replace "null" string with None for max_depth
rf_param_grid["max_depth"] = [
    None if v == "null" else v for v in rf_param_grid["max_depth"]
]

print("🔍 GridSearch for Random Forest...")
print(f"   Grid: {rf_param_grid}")
print("-" * 60)

rf_model, rf_best_params = grid_search_rf(
    X_train, y_train,
    param_grid=rf_param_grid,
    random_state=RANDOM_STATE,
)

In [ ]:
# CV result with best params
from sklearn.ensemble import RandomForestClassifier

print("\n📊 Cross-validation with best RF params:")
rf_cv_model = RandomForestClassifier(**rf_best_params, n_jobs=-1, random_state=RANDOM_STATE)
rf_cv_result = cross_validate_model(rf_cv_model, X_train, y_train,
                                     model_name="Random Forest", random_state=RANDOM_STATE)

# Training AUC sanity check
rf_train_auc = roc_auc_score(y_train, rf_model.predict_proba(X_train)[:, 1])
print(f"   Training AUC-ROC (sanity): {rf_train_auc:.4f}")

# Save
rf_path = OUT_DIR / "model_rf.joblib"
joblib.dump(rf_model, rf_path, compress=3)
print(f"✅ Random Forest saved → {rf_path}")

## 4 · Logistic Regression — GridSearchCV

In [ ]:
from train import grid_search_lr

lr_param_grid = cfg["models"]["lr"]["param_grid"]

print("🔍 GridSearch for Logistic Regression...")
print(f"   Grid: {lr_param_grid}")
print("-" * 60)

lr_model, lr_best_params = grid_search_lr(
    X_train, y_train,
    param_grid=lr_param_grid,
    random_state=RANDOM_STATE,
)

In [ ]:
from sklearn.linear_model import LogisticRegression

print("\n📊 Cross-validation with best LR params:")
lr_cv_model = LogisticRegression(**lr_best_params, solver="lbfgs",
                                   max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE)
lr_cv_result = cross_validate_model(lr_cv_model, X_train, y_train,
                                     model_name="Logistic Regression", random_state=RANDOM_STATE)

lr_train_auc = roc_auc_score(y_train, lr_model.predict_proba(X_train)[:, 1])
print(f"   Training AUC-ROC (sanity): {lr_train_auc:.4f}")

lr_path = OUT_DIR / "model_lr.joblib"
joblib.dump(lr_model, lr_path, compress=3)
print(f"✅ Logistic Regression saved → {lr_path}")

## 5 · CV Results Summary

In [ ]:
cv_results = {
    "lgbm": lgbm_cv_result,
    "rf":   rf_cv_result,
    "lr":   lr_cv_result,
}

# Pretty-print comparison table
print("\n" + "=" * 65)
print("  CROSS-VALIDATION RESULTS SUMMARY (5-fold, optimised on F1)")
print("=" * 65)
print(f"  {'Model':20s}  {'AUC-ROC':>9s}  {'±':>5s}  {'F1':>7s}  {'±':>5s}")
print(f"  {'─'*20}  {'─'*9}  {'─'*5}  {'─'*7}  {'─'*5}")
for name, res in cv_results.items():
    print(f"  {res['model_name']:20s}  {res['mean_auc']:9.4f}  "
          f"{res['std_auc']:5.4f}  {res['mean_f1']:7.4f}  {res['std_f1']:5.4f}")
print("=" * 65)

# Bar chart: CV F1 comparison
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
names = [r["model_name"] for r in cv_results.values()]
aucs  = [r["mean_auc"]   for r in cv_results.values()]
f1s   = [r["mean_f1"]    for r in cv_results.values()]
auc_stds = [r["std_auc"] for r in cv_results.values()]
f1_stds  = [r["std_f1"]  for r in cv_results.values()]
colors   = ["#E63946", "#457B9D", "#2A9D8F"]

axes[0].bar(names, aucs, yerr=auc_stds, color=colors, edgecolor="white",
            capsize=4, alpha=0.85)
axes[0].set_ylim(0.5, 1.0)
axes[0].set_title("CV AUC-ROC (5-fold)", fontweight="bold")
axes[0].set_ylabel("AUC-ROC")
for i, (v, e) in enumerate(zip(aucs, auc_stds)):
    axes[0].text(i, v + e + 0.005, f"{v:.3f}", ha="center", fontsize=10)

axes[1].bar(names, f1s, yerr=f1_stds, color=colors, edgecolor="white",
            capsize=4, alpha=0.85)
axes[1].set_ylim(0.0, 1.0)
axes[1].set_title("CV F1 — Failure Class (5-fold)", fontweight="bold")
axes[1].set_ylabel("F1 Score")
for i, (v, e) in enumerate(zip(f1s, f1_stds)):
    axes[1].text(i, v + e + 0.005, f"{v:.3f}", ha="center", fontsize=10)

plt.suptitle("Cross-Validation Results Comparison", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(OUT_DIR / "cv_results_comparison.png", bbox_inches="tight", dpi=150)
plt.show()

# Save CV results JSON
with open(OUT_DIR / "cv_results.json", "w") as f:
    # Remove non-serialisable fields
    save_cv = {k: {kk: vv for kk, vv in v.items() if not isinstance(vv, np.ndarray)}
               for k, v in cv_results.items()}
    json.dump(save_cv, f, indent=2, default=float)
print(f"\n✅ CV results saved → {OUT_DIR / 'cv_results.json'}")

In [ ]:
print("=" * 70)
print("  NOTEBOOK 04 COMPLETE")
print("=" * 70)
print(f"  model_lgbm.joblib → {OUT_DIR / 'model_lgbm.joblib'}")
print(f"  model_rf.joblib   → {OUT_DIR / 'model_rf.joblib'}")
print(f"  model_lr.joblib   → {OUT_DIR / 'model_lr.joblib'}")
print("\n✅ Proceed to 05_model_evaluation.ipynb")